# Cuaderno 7: Pruebas de Hipótesis
## Diplomado en Ciencia de Datos - Clase de 2.5 horas

**Objetivos de la clase:**
- Comprender la lógica de la inferencia estadística y las pruebas de hipótesis.
- Distinguir entre hipótesis nula y alternativa.
- Conocer los tipos de errores (Tipo I y Tipo II).
- Aplicar pruebas paramétricas y no paramétricas en Python.
- Interpretar correctamente los resultados (p-valor, intervalo de confianza).
- Identificar supuestos y limitaciones.

## 1. Introducción a las Pruebas de Hipótesis (15 min)

### ¿Qué es una prueba de hipótesis?
Es un procedimiento estadístico para tomar decisiones sobre una población basado en una muestra aleatoria.

### Componentes clave:
- **Hipótesis nula (H₀):** Afirmación que se asume verdadera (ej: media = 100, no hay efecto).
- **Hipótesis alternativa (H₁ o Hₐ):** Afirmación que queremos evidenciar (ej: media ≠ 100, hay efecto).
- **Estadístico de prueba:** Valor calculado a partir de los datos.
- **p-valor:** Probabilidad de obtener un resultado tan extremo como el observado si H₀ es cierta.
- **Nivel de significancia (α):** Umbral para rechazar H₀ (común: 0.05).
- **Regla de decisión:** Si p-valor < α → rechazar H₀.

### Tipos de hipótesis según la dirección:
- Bilateral (dos colas): H₁: μ ≠ μ₀
- Unilateral izquierda: H₁: μ < μ₀
- Unilateral derecha: H₁: μ > μ₀

### Errores posibles:
| Decisión / Realidad | H₀ verdadera       | H₀ falsa           |
|---------------------|--------------------|--------------------|
| No rechazar H₀      | Correcto (1-α)     | Error Tipo II (β)  |
| Rechazar H₀         | Error Tipo I (α)   | Correcto (Potencia)|

- **Error Tipo I:** Falso positivo.
- **Error Tipo II:** Falso negativo.
- **Potencia = 1 - β:** Probabilidad de detectar un efecto real.

## 2. Supuestos y cuidados generales (10 min)

- **Muestreo aleatorio:** La muestra debe representar a la población.
- **Independencia:** Las observaciones deben ser independientes (n < 10% de la población si es sin reemplazo).
- **Normalidad (para pruebas paramétricas):** Los datos o las medias muestrales deben seguir una distribución normal (Teorema Central del Límite ayuda si n ≥ 30).
- **Homogeneidad de varianzas** (para pruebas como t de Student para dos muestras).
- **Cuidado con:** Datos atípicos, muestras pequeñas, pruebas múltiples (corrección de Bonferroni), interpretación del p-valor como 'tamaño del efecto'.

## 3. Implementación en Python: Librerías necesarias

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import TTestPower, NormalIndPower
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import GofChisquarePower

# Configuración de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 4. Ejemplo 1: Prueba t para una muestra (15 min)

**Contexto:** Un fabricante afirma que sus barras de cereal pesan en promedio 50 g. Tomamos una muestra de 30 barras y queremos verificar si el peso promedio es diferente.

In [ ]:
# Generar datos simulados
np.random.seed(123)
pesos_reales = np.random.normal(loc=49.5, scale=2, size=30)  # media real 49.5

# Prueba t de una muestra
t_stat, p_valor = stats.ttest_1samp(pesos_reales, popmean=50)

# Resultados
print(f"Estadístico t: {t_stat:.4f}")
print(f"p-valor: {p_valor:.4f}")

# Decisión
alpha = 0.05
if p_valor < alpha:
    print("Rechazamos H₀: el peso promedio es diferente a 50 g.")
else:
    print("No rechazamos H₀: no hay evidencia suficiente.")

# Interpretación adicional
print("\nMedia muestral:", np.mean(pesos_reales))
print("IC 95% para la media:", stats.t.interval(0.95, df=len(pesos_reales)-1, loc=np.mean(pesos_reales), scale=stats.sem(pesos_reales)))

## 5. Ejemplo 2: Prueba t para dos muestras independientes (20 min)

**Contexto:** Comparar si el tiempo de carga de dos versiones de una app (A y B) es diferente.

In [ ]:
# Datos simulados
np.random.seed(456)
tiempo_A = np.random.normal(loc=2.5, scale=0.3, size=50)  # media 2.5 seg
tiempo_B = np.random.normal(loc=2.7, scale=0.3, size=50)  # media 2.7 seg

# Prueba t de Welch (no asume varianzas iguales)
t_stat, p_valor = stats.ttest_ind(tiempo_A, tiempo_B, equal_var=False)

print(f"t = {t_stat:.3f}, p-valor = {p_valor:.4f}")
if p_valor < 0.05:
    print("Diferencia significativa: la versión B es más lenta.")
else:
    print("No hay diferencia significativa.")

### Verificación de supuestos

In [ ]:
# Gráficos de normalidad
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(tiempo_A, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribución - Grupo A')
sns.histplot(tiempo_B, kde=True, ax=axes[1], color='red')
axes[1].set_title('Distribución - Grupo B')
plt.tight_layout()
plt.show()

# Prueba de Levene para homogeneidad de varianzas
stat_levene, p_levene = stats.levene(tiempo_A, tiempo_B)
print(f"Prueba de Levene: p-valor = {p_levene:.4f}")
if p_levene < 0.05:
    print("Las varianzas son significativamente diferentes → usar Welch.")
else:
    print("No hay evidencia de diferencia de varianzas.")

## 6. Ejemplo 3: Prueba t para muestras pareadas (15 min)

**Contexto:** Medir el peso de 20 personas antes y después de un programa de ejercicio de 4 semanas.

In [ ]:
np.random.seed(789)
antes = np.random.normal(loc=80, scale=10, size=20)
despues = antes - np.random.normal(loc=2, scale=1.5, size=20)  # pérdida media 2 kg

# Prueba t pareada
t_stat, p_valor = stats.ttest_rel(antes, despues)

print(f"t = {t_stat:.3f}, p-valor = {p_valor:.5f}")
if p_valor < 0.05:
    print("Hay una reducción significativa del peso.")
else:
    print("No se detectó cambio significativo.")

## 7. Ejemplo 4: Prueba de proporciones (15 min)

**Contexto:** En una campaña política, el 42% de 400 encuestados apoya al candidato X. ¿Podemos decir que el apoyo real es superior al 40%?

In [ ]:
# Datos
n = 400
exitos = 168  # 42% de 400
p0 = 0.40

# Prueba z de una proporción (unilateral derecha)
z_stat, p_valor = proportions_ztest(count=exitos, nobs=n, value=p0, alternative='larger')

print(f"z = {z_stat:.3f}, p-valor = {p_valor:.4f}")
if p_valor < 0.05:
    print("Rechazamos H₀: el apoyo es significativamente mayor al 40%.")
else:
    print("No hay evidencia suficiente.")

# Comparación de dos proporciones (ejemplo A/B test)
print("\n--- Comparación de dos proporciones ---")
conversiones = [45, 32]  # grupo A, grupo B
tamaños = [500, 500]

z_stat, p_valor = proportions_ztest(count=conversiones, nobs=tamaños, alternative='two-sided')
print(f"Diferencia de proporciones: p-valor = {p_valor:.4f}")

## 8. Ejemplo 5: Prueba Chi-cuadrado de independencia (15 min)

**Contexto:** Relación entre género (Hombre/Mujer) y preferencia por un producto (Sí/No).

In [ ]:
# Tabla de contingencia
datos = np.array([[30, 20],   # Hombres: Sí=30, No=20
                  [25, 35]])  # Mujeres: Sí=25, No=35

# Prueba chi-cuadrado
chi2, p_valor, dof, expected = stats.chi2_contingency(datos)

print(f"Chi-cuadrado = {chi2:.3f}")
print(f"p-valor = {p_valor:.4f}")
print("Frecuencias esperadas:\n", expected)

if p_valor < 0.05:
    print("Hay asociación significativa entre género y preferencia.")
else:
    print("No hay evidencia de asociación.")

## 9. Pruebas no paramétricas (alternativas robustas) (15 min)

### ¿Cuándo usarlas?
- Datos no normales.
- Muestras pequeñas.
- Variables ordinales o con outliers.

### Ejemplo: Mann-Whitney U (alternativa a t para dos muestras independientes)

In [ ]:
# Datos no normales
np.random.seed(321)
grupo1 = np.random.exponential(scale=2, size=30)
grupo2 = np.random.exponential(scale=3, size=30)

# Prueba U de Mann-Whitney
u_stat, p_valor = stats.mannwhitneyu(grupo1, grupo2, alternative='two-sided')
print(f"Mann-Whitney U: p-valor = {p_valor:.4f}")

## 10. Tamaño del efecto (10 min)

No basta con p-valor pequeño; hay que medir la magnitud del efecto.

### d de Cohen (diferencia de medias estandarizada)

In [ ]:
def cohen_d(x, y):
    n1, n2 = len(x), len(y)
    var_pooled = ((n1-1)*np.var(x, ddof=1) + (n2-1)*np.var(y, ddof=1)) / (n1+n2-2)
    return (np.mean(x) - np.mean(y)) / np.sqrt(var_pooled)

d = cohen_d(tiempo_A, tiempo_B)
print(f"d de Cohen = {d:.3f}")
print("Interpretación: pequeño=0.2, mediano=0.5, grande=0.8")

## 11. Cálculo de potencia y tamaño de muestra (15 min)

Planificar un estudio antes de recolectar datos.

In [ ]:
# Potencia para una prueba t de dos muestras
efecto = 0.5  # d de Cohen esperado
alpha = 0.05
n_por_grupo = 50

potencia = TTestPower().power(effect_size=efecto, nobs=n_por_grupo, alpha=alpha)
print(f"Potencia = {potencia:.3f}")

# Tamaño de muestra necesario para potencia 0.8
n_necesario = TTestPower().solve_power(effect_size=efecto, power=0.8, alpha=alpha)
print(f"N por grupo necesario = {np.ceil(n_necesario):.0f}")

## 12. Errores comunes y buenas prácticas (10 min)

### ❌ Malas prácticas:
- Buscar un p-valor < 0.05 a toda costa (p-hacking).
- No verificar supuestos.
- Interpretar p-valor como probabilidad de que H₀ sea cierta.
- Reportar solo p-valor sin tamaño del efecto.

### ✅ Buenas prácticas:
- Pre-registrar el análisis.
- Reportar intervalo de confianza y tamaño del efecto.
- Usar correcciones por pruebas múltiples (Bonferroni, FDR).
- Realizar análisis de sensibilidad.
- Usar bootstrapping si los supuestos fallan.

### Ejemplo de corrección de Bonferroni:

In [ ]:
p_valores = [0.01, 0.04, 0.03, 0.20]  # 4 pruebas
rechazar, p_ajustados, _, _ = multipletests(p_valores, alpha=0.05, method='bonferroni')
print("P-valores originales:", p_valores)
print("P-valores ajustados:", p_ajustados)
print("Rechazar H₀ después de ajuste:", rechazar)

## 13. Ejercicio integrador (15 min)

**Problema:** Un equipo de marketing quiere saber si una nueva landing page (B) aumenta la tasa de conversión respecto a la actual (A). Se recolectaron datos de 1000 visitantes por página.

- A: 120 conversiones de 1000 (12%)
- B: 150 conversiones de 1000 (15%)

1. Plantea H₀ y H₁.
2. Realiza la prueba de proporciones.
3. Calcula el tamaño del efecto (diferencia de proporciones).
4. Verifica si la potencia es suficiente (asume α=0.05, tamaño de efecto observado).
5. Concluye con una recomendación de negocio.

In [ ]:
# Solución
conversion_A = 120
conversion_B = 150
n_A = n_B = 1000

# Prueba unilateral derecha (B > A)
z_stat, p_valor = proportions_ztest([conversion_B, conversion_A], [n_B, n_A], alternative='larger')
print(f"z = {z_stat:.3f}, p-valor = {p_valor:.4f}")

# Tamaño del efecto (diferencia absoluta)
efecto = (conversion_B/n_B) - (conversion_A/n_A)
print(f"Diferencia = {efecto:.3f} ({efecto*100:.1f} pp)")

# Potencia post-hoc
n_total = n_A + n_B
potencia = GofChisquarePower().power(effect_size=efecto, nobs=n_total, alpha=0.05)
print(f"Potencia observada: {potencia:.3f}")

# Conclusión
if p_valor < 0.05:
    print("\n✅ Conclusión: Rechazamos H₀. La nueva landing page aumenta significativamente la conversión.")
else:
    print("\n❌ Conclusión: No hay evidencia suficiente para afirmar que la nueva página es mejor.")

## 14. Cierre y recursos adicionales (5 min)

### Conclusiones clave:
- Las pruebas de hipótesis son herramientas para tomar decisiones bajo incertidumbre.
- Siempre verificar supuestos.
- No olvidar el contexto del problema (significancia estadística ≠ relevancia práctica).
- Reportar: estadístico, p-valor, intervalo de confianza y tamaño del efecto.

### Recursos:
- [StatQuest con Josh Starmer (YouTube)](https://www.youtube.com/user/joshstarmer)
- Libro: "OpenIntro Statistics"
- Python: documentación de `scipy.stats` y `statsmodels`

### Preguntas para reflexión:
1. ¿Qué pasaría si repetimos una prueba 100 veces con α=0.05?
2. ¿Por qué el p-valor no es la probabilidad de que H₀ sea cierta?
3. ¿Cuándo preferirías una prueba no paramétrica sobre una paramétrica?

---
**Fin de la clase**